# EEG Data Exploration and Analysis

This notebook provides comprehensive exploration and analysis of EEG datasets for stress detection.

**Authors:** Youssef Mahdi, Hamza El Haiki  
**Date:** July 2024  
**Project:** Predicting Stress Peaks based on EEG Signals by Hybrid Approach of CNN-LSTM

## Objectives

1. Load and explore multiple EEG datasets
2. Analyze data quality and characteristics
3. Visualize EEG signals and patterns
4. Identify stress-related features
5. Prepare data for preprocessing pipeline

## Setup and Imports

In [ ]:
# Import required libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Add project root to path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

# Import project modules
from src.data.data_loader import EEGDataLoader
from src.preprocessing.eeg_preprocessor import EEGPreprocessor
from src.utils.config import get_config, DATASET_CONFIG, PREPROCESSING_CONFIG
from src.utils.visualization import EEGVisualizer

print("All imports successful!")
print(f"Project root: {project_root}")

## Configuration and Setup

In [ ]:
# Initialize components
data_loader = EEGDataLoader()
preprocessor = EEGPreprocessor(**PREPROCESSING_CONFIG)
visualizer = EEGVisualizer()

# Display configuration
print("Dataset Configuration:")
for key, value in DATASET_CONFIG.items():
    print(f"  {key}: {value}")

print("\nPreprocessing Configuration:")
for key, value in PREPROCESSING_CONFIG.items():
    print(f"  {key}: {value}")

## Dataset Loading and Basic Information

In [ ]:
# Load all available datasets
print("Loading EEG datasets...")
datasets = data_loader.load_all_datasets()

print(f"\nLoaded {len(datasets)} datasets:")
for name, data in datasets.items():
    print(f"\n{name}:")
    print(f"  Shape: {data['raw_data'].shape}")
    print(f"  Sampling Rate: {data.get('sampling_rate', 'Unknown')} Hz")
    print(f"  Duration: {data['raw_data'].shape[0] / data.get('sampling_rate', 256):.2f} seconds")
    print(f"  Channels: {data['raw_data'].shape[1] if len(data['raw_data'].shape) > 1 else 1}")
    if 'metadata' in data:
        print(f"  Metadata keys: {list(data['metadata'].keys())}")

## Dataset Summary Statistics

In [ ]:
# Create dataset summary
summary_data = []

for name, data in datasets.items():
    raw_data = data['raw_data']
    
    summary = {
        'Dataset': name,
        'Samples': raw_data.shape[0],
        'Channels': raw_data.shape[1] if len(raw_data.shape) > 1 else 1,
        'Duration (s)': raw_data.shape[0] / data.get('sampling_rate', 256),
        'Sampling Rate (Hz)': data.get('sampling_rate', 'Unknown'),
        'Mean Amplitude': np.mean(raw_data),
        'Std Amplitude': np.std(raw_data),
        'Min Amplitude': np.min(raw_data),
        'Max Amplitude': np.max(raw_data),
        'Data Range': np.max(raw_data) - np.min(raw_data)
    }
    summary_data.append(summary)

summary_df = pd.DataFrame(summary_data)
print("Dataset Summary:")
print(summary_df.to_string(index=False))

## Data Quality Assessment

In [ ]:
# Check for missing values and data quality issues
print("Data Quality Assessment:")
print("=" * 50)

for name, data in datasets.items():
    raw_data = data['raw_data']
    
    print(f"\n{name}:")
    
    # Missing values
    missing_count = np.isnan(raw_data).sum()
    missing_percent = (missing_count / raw_data.size) * 100
    print(f"  Missing values: {missing_count} ({missing_percent:.2f}%)")
    
    # Infinite values
    inf_count = np.isinf(raw_data).sum()
    print(f"  Infinite values: {inf_count}")
    
    # Zero values (potential artifacts)
    zero_count = (raw_data == 0).sum()
    zero_percent = (zero_count / raw_data.size) * 100
    print(f"  Zero values: {zero_count} ({zero_percent:.2f}%)")
    
    # Outliers (values beyond 3 standard deviations)
    mean_val = np.mean(raw_data)
    std_val = np.std(raw_data)
    outliers = np.abs(raw_data - mean_val) > 3 * std_val
    outlier_count = outliers.sum()
    outlier_percent = (outlier_count / raw_data.size) * 100
    print(f"  Outliers (>3σ): {outlier_count} ({outlier_percent:.2f}%)")
    
    # Dynamic range
    dynamic_range = np.max(raw_data) - np.min(raw_data)
    print(f"  Dynamic range: {dynamic_range:.2f}")

## EEG Signal Visualization

In [ ]:
# Visualize raw EEG signals from each dataset
fig, axes = plt.subplots(len(datasets), 1, figsize=(15, 4 * len(datasets)))
if len(datasets) == 1:
    axes = [axes]

for idx, (name, data) in enumerate(datasets.items()):
    raw_data = data['raw_data']
    sampling_rate = data.get('sampling_rate', 256)
    
    # Select first 4 seconds of data
    duration = min(4.0, raw_data.shape[0] / sampling_rate)
    samples_to_plot = int(duration * sampling_rate)
    
    if len(raw_data.shape) > 1:
        # Multi-channel data - plot first few channels
        channels_to_plot = min(5, raw_data.shape[1])
        time_axis = np.arange(samples_to_plot) / sampling_rate
        
        for ch in range(channels_to_plot):
            axes[idx].plot(time_axis, raw_data[:samples_to_plot, ch] + ch * 100, 
                          label=f'Channel {ch+1}', alpha=0.8)
    else:
        # Single-channel data
        time_axis = np.arange(samples_to_plot) / sampling_rate
        axes[idx].plot(time_axis, raw_data[:samples_to_plot], 'b-', alpha=0.8)
    
    axes[idx].set_title(f'Raw EEG Signals - {name}')
    axes[idx].set_xlabel('Time (seconds)')
    axes[idx].set_ylabel('Amplitude (μV)')
    axes[idx].grid(True, alpha=0.3)
    if len(raw_data.shape) > 1 and raw_data.shape[1] <= 5:
        axes[idx].legend()

plt.tight_layout()
plt.show()

## Frequency Domain Analysis

In [ ]:
# Analyze frequency content of EEG signals
frequency_bands = PREPROCESSING_CONFIG['feature_extraction']['frequency_bands']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

dataset_names = list(datasets.keys())[:4]  # Analyze first 4 datasets

for idx, name in enumerate(dataset_names):
    if idx >= len(axes):
        break
        
    data = datasets[name]
    raw_data = data['raw_data']
    sampling_rate = data.get('sampling_rate', 256)
    
    # Select a representative channel
    if len(raw_data.shape) > 1:
        signal_data = raw_data[:, 0]  # First channel
    else:
        signal_data = raw_data
    
    # Compute power spectral density
    freqs, psd = signal.welch(signal_data, fs=sampling_rate, nperseg=1024)
    
    # Plot PSD
    axes[idx].semilogy(freqs, psd, 'b-', alpha=0.8)
    
    # Highlight frequency bands
    colors = ['red', 'orange', 'green', 'blue', 'purple']
    for i, (band_name, (low, high)) in enumerate(frequency_bands.items()):
        mask = (freqs >= low) & (freqs <= high)
        if np.any(mask):
            axes[idx].fill_between(freqs[mask], psd[mask], alpha=0.3, 
                                 color=colors[i % len(colors)], label=band_name)
    
    axes[idx].set_title(f'Power Spectral Density - {name}')
    axes[idx].set_xlabel('Frequency (Hz)')
    axes[idx].set_ylabel('Power Spectral Density')
    axes[idx].set_xlim(0, 50)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].legend()

plt.tight_layout()
plt.show()

## Frequency Band Power Analysis

In [ ]:
# Calculate power in different frequency bands
band_powers = {}

for name, data in datasets.items():
    raw_data = data['raw_data']
    sampling_rate = data.get('sampling_rate', 256)
    
    # Select representative channel
    if len(raw_data.shape) > 1:
        signal_data = raw_data[:, 0]
    else:
        signal_data = raw_data
    
    # Compute power in each frequency band
    freqs, psd = signal.welch(signal_data, fs=sampling_rate, nperseg=1024)
    
    band_power = {}
    for band_name, (low, high) in frequency_bands.items():
        mask = (freqs >= low) & (freqs <= high)
        power = np.trapz(psd[mask], freqs[mask]) if np.any(mask) else 0
        band_power[band_name] = power
    
    band_powers[name] = band_power

# Create DataFrame for visualization
band_power_df = pd.DataFrame(band_powers).T
print("Frequency Band Powers:")
print(band_power_df)

# Visualize band powers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Stacked bar chart
band_power_df.plot(kind='bar', stacked=True, ax=ax1, colormap='viridis')
ax1.set_title('Frequency Band Power Distribution by Dataset')
ax1.set_xlabel('Dataset')
ax1.set_ylabel('Power')
ax1.legend(title='Frequency Bands')
ax1.tick_params(axis='x', rotation=45)

# Normalized percentage chart
band_power_normalized = band_power_df.div(band_power_df.sum(axis=1), axis=0) * 100
band_power_normalized.plot(kind='bar', stacked=True, ax=ax2, colormap='viridis')
ax2.set_title('Normalized Frequency Band Power Distribution')
ax2.set_xlabel('Dataset')
ax2.set_ylabel('Percentage (%)')
ax2.legend(title='Frequency Bands')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Statistical Analysis

In [ ]:
# Statistical analysis of EEG signals
stats_data = []

for name, data in datasets.items():
    raw_data = data['raw_data']
    
    # Flatten if multi-channel
    if len(raw_data.shape) > 1:
        signal_data = raw_data.flatten()
    else:
        signal_data = raw_data
    
    # Calculate statistics
    stats_dict = {
        'Dataset': name,
        'Mean': np.mean(signal_data),
        'Median': np.median(signal_data),
        'Std': np.std(signal_data),
        'Variance': np.var(signal_data),
        'Skewness': stats.skew(signal_data),
        'Kurtosis': stats.kurtosis(signal_data),
        'Min': np.min(signal_data),
        'Max': np.max(signal_data),
        'Range': np.max(signal_data) - np.min(signal_data),
        'IQR': np.percentile(signal_data, 75) - np.percentile(signal_data, 25)
    }
    stats_data.append(stats_dict)

stats_df = pd.DataFrame(stats_data)
print("Statistical Summary:")
print(stats_df.round(4))

## Data Distribution Analysis

In [ ]:
# Analyze amplitude distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, data) in enumerate(datasets.items()):
    if idx >= len(axes):
        break
        
    raw_data = data['raw_data']
    
    # Flatten if multi-channel
    if len(raw_data.shape) > 1:
        signal_data = raw_data.flatten()
    else:
        signal_data = raw_data
    
    # Plot histogram
    axes[idx].hist(signal_data, bins=50, alpha=0.7, density=True, color='skyblue', edgecolor='black')
    
    # Overlay normal distribution for comparison
    mu, sigma = np.mean(signal_data), np.std(signal_data)
    x = np.linspace(np.min(signal_data), np.max(signal_data), 100)
    normal_curve = stats.norm.pdf(x, mu, sigma)
    axes[idx].plot(x, normal_curve, 'r-', linewidth=2, label='Normal Distribution')
    
    axes[idx].set_title(f'Amplitude Distribution - {name}')
    axes[idx].set_xlabel('Amplitude')
    axes[idx].set_ylabel('Density')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Correlation Analysis (Multi-channel data)

In [ ]:
# Analyze correlations between channels for multi-channel datasets
multichannel_datasets = {name: data for name, data in datasets.items() 
                        if len(data['raw_data'].shape) > 1 and data['raw_data'].shape[1] > 1}

if multichannel_datasets:
    fig, axes = plt.subplots(1, len(multichannel_datasets), figsize=(5 * len(multichannel_datasets), 5))
    if len(multichannel_datasets) == 1:
        axes = [axes]
    
    for idx, (name, data) in enumerate(multichannel_datasets.items()):
        raw_data = data['raw_data']
        
        # Calculate correlation matrix
        # Use a subset of data for computation efficiency
        subset_size = min(10000, raw_data.shape[0])
        correlation_matrix = np.corrcoef(raw_data[:subset_size].T)
        
        # Plot correlation heatmap
        im = axes[idx].imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        axes[idx].set_title(f'Channel Correlation - {name}')
        axes[idx].set_xlabel('Channel')
        axes[idx].set_ylabel('Channel')
        
        # Add colorbar
        plt.colorbar(im, ax=axes[idx], label='Correlation')
    
    plt.tight_layout()
    plt.show()
    
    # Print correlation statistics
    print("\nChannel Correlation Statistics:")
    for name, data in multichannel_datasets.items():
        raw_data = data['raw_data']
        subset_size = min(10000, raw_data.shape[0])
        correlation_matrix = np.corrcoef(raw_data[:subset_size].T)
        
        # Get upper triangle (excluding diagonal)
        upper_triangle = correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]
        
        print(f"\n{name}:")
        print(f"  Mean correlation: {np.mean(upper_triangle):.3f}")
        print(f"  Max correlation: {np.max(upper_triangle):.3f}")
        print(f"  Min correlation: {np.min(upper_triangle):.3f}")
        print(f"  Std correlation: {np.std(upper_triangle):.3f}")
else:
    print("No multi-channel datasets found for correlation analysis.")

## Preliminary Feature Extraction

In [ ]:
# Extract basic features from each dataset
print("Extracting preliminary features...")

feature_data = []

for name, data in datasets.items():
    raw_data = data['raw_data']
    sampling_rate = data.get('sampling_rate', 256)
    
    try:
        # Preprocess data first
        processed_data = preprocessor.preprocess_pipeline(raw_data, sampling_rate)
        
        # Extract features
        features = preprocessor.extract_all_features(processed_data)
        
        feature_info = {
            'Dataset': name,
            'Feature_Count': features.shape[1] if len(features.shape) > 1 else len(features),
            'Sample_Count': features.shape[0] if len(features.shape) > 1 else 1,
            'Feature_Mean': np.mean(features),
            'Feature_Std': np.std(features),
            'Feature_Range': np.max(features) - np.min(features)
        }
        feature_data.append(feature_info)
        
        print(f"✓ {name}: {feature_info['Feature_Count']} features extracted")
        
    except Exception as e:
        print(f"✗ {name}: Feature extraction failed - {str(e)}")
        continue

if feature_data:
    feature_df = pd.DataFrame(feature_data)
    print("\nFeature Extraction Summary:")
    print(feature_df)
else:
    print("No features could be extracted from any dataset.")

## Dimensionality Reduction and Visualization

In [ ]:
# Apply PCA and t-SNE for feature visualization
if feature_data:
    print("Applying dimensionality reduction...")
    
    # Collect all features for combined analysis
    all_features = []
    all_labels = []
    
    for name, data in datasets.items():
        try:
            raw_data = data['raw_data']
            sampling_rate = data.get('sampling_rate', 256)
            
            processed_data = preprocessor.preprocess_pipeline(raw_data, sampling_rate)
            features = preprocessor.extract_all_features(processed_data)
            
            # Ensure 2D array
            if len(features.shape) == 1:
                features = features.reshape(1, -1)
            
            all_features.append(features)
            all_labels.extend([name] * features.shape[0])
            
        except Exception as e:
            print(f"Skipping {name}: {str(e)}")
            continue
    
    if all_features:
        # Combine all features
        combined_features = np.vstack(all_features)
        
        print(f"Combined features shape: {combined_features.shape}")
        
        # Apply PCA
        pca = PCA(n_components=2)
        features_pca = pca.fit_transform(combined_features)
        
        # Apply t-SNE (on subset if too large)
        if combined_features.shape[0] > 1000:
            # Use subset for t-SNE
            indices = np.random.choice(combined_features.shape[0], 1000, replace=False)
            features_subset = combined_features[indices]
            labels_subset = [all_labels[i] for i in indices]
        else:
            features_subset = combined_features
            labels_subset = all_labels
        
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features_subset)-1))
        features_tsne = tsne.fit_transform(features_subset)
        
        # Plot results
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # PCA plot
        unique_labels = list(set(all_labels))
        colors = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))
        
        for i, label in enumerate(unique_labels):
            mask = np.array(all_labels) == label
            ax1.scatter(features_pca[mask, 0], features_pca[mask, 1], 
                       c=[colors[i]], label=label, alpha=0.6, s=20)
        
        ax1.set_title(f'PCA Visualization\n(Explained variance: {pca.explained_variance_ratio_.sum():.2f})')
        ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2f})')
        ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2f})')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # t-SNE plot
        for i, label in enumerate(unique_labels):
            mask = np.array(labels_subset) == label
            ax2.scatter(features_tsne[mask, 0], features_tsne[mask, 1], 
                       c=[colors[i]], label=label, alpha=0.6, s=20)
        
        ax2.set_title('t-SNE Visualization')
        ax2.set_xlabel('t-SNE 1')
        ax2.set_ylabel('t-SNE 2')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nPCA explained variance ratio: {pca.explained_variance_ratio_}")
        print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.3f}")
    else:
        print("No features available for dimensionality reduction.")
else:
    print("No feature data available for dimensionality reduction.")

## Data Quality and Preprocessing Recommendations

In [ ]:
# Generate data quality report and preprocessing recommendations
print("DATA QUALITY REPORT AND RECOMMENDATIONS")
print("=" * 60)

for name, data in datasets.items():
    raw_data = data['raw_data']
    sampling_rate = data.get('sampling_rate', 256)
    
    print(f"\n{name.upper()}:")
    print("-" * len(name))
    
    # Data characteristics
    print(f"Shape: {raw_data.shape}")
    print(f"Duration: {raw_data.shape[0] / sampling_rate:.2f} seconds")
    print(f"Sampling Rate: {sampling_rate} Hz")
    
    # Quality metrics
    missing_percent = (np.isnan(raw_data).sum() / raw_data.size) * 100
    zero_percent = ((raw_data == 0).sum() / raw_data.size) * 100
    
    mean_val = np.mean(raw_data)
    std_val = np.std(raw_data)
    outlier_percent = ((np.abs(raw_data - mean_val) > 3 * std_val).sum() / raw_data.size) * 100
    
    print(f"Missing values: {missing_percent:.2f}%")
    print(f"Zero values: {zero_percent:.2f}%")
    print(f"Outliers (>3σ): {outlier_percent:.2f}%")
    print(f"Dynamic range: {np.max(raw_data) - np.min(raw_data):.2f}")
    
    # Recommendations
    print("\nRecommendations:")
    
    if missing_percent > 5:
        print("  ⚠️  HIGH missing values - consider interpolation or data exclusion")
    elif missing_percent > 0:
        print("  ⚡ Some missing values - apply interpolation")
    else:
        print("  ✅ No missing values")
    
    if zero_percent > 10:
        print("  ⚠️  HIGH zero values - check for recording artifacts")
    elif zero_percent > 5:
        print("  ⚡ Some zero values - investigate potential artifacts")
    else:
        print("  ✅ Acceptable zero value rate")
    
    if outlier_percent > 5:
        print("  ⚠️  HIGH outlier rate - apply robust preprocessing")
    elif outlier_percent > 1:
        print("  ⚡ Some outliers - consider outlier removal")
    else:
        print("  ✅ Low outlier rate")
    
    if sampling_rate < 128:
        print("  ⚠️  LOW sampling rate - may limit frequency analysis")
    elif sampling_rate > 512:
        print("  ⚡ HIGH sampling rate - consider downsampling for efficiency")
    else:
        print("  ✅ Appropriate sampling rate")
    
    # Specific preprocessing suggestions
    print("\nSuggested preprocessing steps:")
    print("  1. Bandpass filtering (0.5-50 Hz)")
    print("  2. Notch filtering (50/60 Hz)")
    print("  3. ICA for artifact removal")
    print("  4. Feature extraction (time/frequency domain)")
    print("  5. Normalization/standardization")

print("\n" + "=" * 60)
print("OVERALL DATASET SUITABILITY FOR STRESS DETECTION:")
print("=" * 60)

total_datasets = len(datasets)
usable_datasets = len([d for d in datasets.values() if d['raw_data'].size > 0])

print(f"Total datasets loaded: {total_datasets}")
print(f"Usable datasets: {usable_datasets}")
print(f"Success rate: {(usable_datasets/total_datasets)*100:.1f}%")

if usable_datasets >= 3:
    print("\n✅ EXCELLENT: Multiple datasets available for robust model training")
elif usable_datasets >= 2:
    print("\n⚡ GOOD: Sufficient datasets for model development")
elif usable_datasets >= 1:
    print("\n⚠️  MINIMAL: Limited data - consider acquiring additional datasets")
else:
    print("\n❌ INSUFFICIENT: No usable datasets - data acquisition required")

## Summary and Next Steps

In [ ]:
print("EEG DATA EXPLORATION SUMMARY")
print("=" * 50)

print("\n📊 DATASETS ANALYZED:")
for name, data in datasets.items():
    shape = data['raw_data'].shape
    duration = shape[0] / data.get('sampling_rate', 256)
    print(f"  • {name}: {shape} | {duration:.1f}s")

print("\n🔬 KEY FINDINGS:")
print("  • Multiple EEG datasets successfully loaded")
print("  • Varying sampling rates and channel configurations")
print("  • Frequency content spans all major EEG bands")
print("  • Feature extraction pipeline functional")
print("  • Data quality varies across datasets")

print("\n⚡ PREPROCESSING PIPELINE READY:")
print("  • Bandpass filtering: 0.5-50 Hz")
print("  • Notch filtering: 50 Hz")
print("  • ICA artifact removal")
print("  • Multi-domain feature extraction")
print("  • Normalization and scaling")

print("\n🎯 NEXT STEPS:")
print("  1. Run preprocessing pipeline on all datasets")
print("  2. Create stress labels using intelligent labeling")
print("  3. Train CNN-LSTM hybrid model")
print("  4. Evaluate model performance")
print("  5. Fine-tune hyperparameters")

print("\n🚀 PROJECT STATUS: READY FOR MODEL TRAINING")
print("=" * 50)